# Manos al código: transformers en Python

**Extensión opcional · 60–90 minutos**

En este cuaderno vas a usar modelos preentrenados para: generar texto, observar una tokenización real, distinguir *embeddings* de activaciones contextuales, comparar oraciones y visualizar activaciones y atención. No entrenaremos ningún modelo.

> Los modelos se descargan de Hugging Face la primera vez. En Colab, elegí **Entorno de ejecución → Ejecutar todas**; CPU alcanza.

## 0. Preparación
Usamos un generador pequeño (`SmolLM2-135M-Instruct`) y un encoder multilingüe (`distilbert-base-multilingual-cased`). Separar ambos ayuda a ver que *usar un modelo de lenguaje* puede significar tareas diferentes.

In [ ]:
%pip -q install "transformers>=4.46,<5" torch scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer, pipeline

torch.manual_seed(7)
print("PyTorch:", torch.__version__)

## 1. Generar texto con `pipeline`
Una `pipeline` reúne tokenizador, modelo y posprocesamiento. La temperatura no cambia lo que el modelo aprendió: modifica el muestreo a partir de sus probabilidades.

In [ ]:
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-135M-Instruct",
    device=-1,  # CPU
)

prompt = "Explicá qué es un token en dos oraciones y con un ejemplo en español."
result = generator(
    prompt, max_new_tokens=70, do_sample=True, temperature=0.7,
    return_full_text=False, pad_token_id=generator.tokenizer.eos_token_id,
)
print(result[0]["generated_text"])

**Probá:** ejecutá la celda anterior varias veces; luego compará `temperature=0.2` y `temperature=1.3`. Con muestreo, una temperatura baja concentra la distribución, pero no obliga por sí sola a elegir siempre el máximo. Para eso se usa generación *greedy*: `do_sample=False`.

## 2. Tokenización y una pasada por el encoder
Pedimos explícitamente estados ocultos y matrices de atención. `model.eval()` desactiva comportamientos de entrenamiento, y `torch.no_grad()` evita calcular gradientes que aquí no necesitamos.

In [ ]:
encoder_name = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(encoder_name)
model = AutoModel.from_pretrained(encoder_name, attn_implementation="eager")
model.eval()

text = "La tokenización puede partir palabras desconocidas."
inputs = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(list(zip(tokens, inputs["input_ids"][0].tolist())))

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True, output_attentions=True)

print("Activación final:", tuple(outputs.last_hidden_state.shape))
print("Estados (embedding + capas):", len(outputs.hidden_states))
print("Matrices de atención:", len(outputs.attentions))

## 3. Embedding de entrada vs. activación contextual
El *embedding de entrada* es la búsqueda inicial asociada al identificador del token. La *activación* o estado oculto es lo que queda después de transformarlo usando el contexto. Ambos son vectores, pero no son intercambiables.

In [ ]:
with torch.no_grad():
    input_embeddings = model.get_input_embeddings()(inputs["input_ids"])
    contextual_activations = outputs.last_hidden_state

position = 3
print("Token elegido:", tokens[position])
print("Primeras 8 coordenadas del embedding:", input_embeddings[0, position, :8])
print("Primeras 8 coordenadas de la activación:", contextual_activations[0, position, :8])
print("Distancia entre ambos:", torch.norm(input_embeddings[0, position] - contextual_activations[0, position]).item())

### El mismo token cambia con el contexto
Buscamos la activación de `banco` en dos oraciones. El identificador de entrada es el mismo; la representación contextual no.

In [ ]:
contexts = ["Me senté en el banco de la plaza.", "El banco aprobó el préstamo."]
batch = tokenizer(contexts, return_tensors="pt", padding=True)
with torch.no_grad():
    contextual = model(**batch).last_hidden_state

for i, sentence in enumerate(contexts):
    pieces = tokenizer.convert_ids_to_tokens(batch["input_ids"][i])
    pos = pieces.index("banco")
    print(sentence, "→ posición", pos, "; norma", contextual[i, pos].norm().item())

v0 = contextual[0, tokenizer.convert_ids_to_tokens(batch["input_ids"][0]).index("banco")]
v1 = contextual[1, tokenizer.convert_ids_to_tokens(batch["input_ids"][1]).index("banco")]
print("Similitud coseno entre los dos 'banco':", torch.nn.functional.cosine_similarity(v0, v1, dim=0).item())

## 4. Un embedding para cada oración
Promediamos las activaciones de los tokens reales (ignorando el *padding*). Es una demostración transparente de *mean pooling*, no necesariamente el mejor método para búsqueda semántica.

In [ ]:
sentences = [
    "El gato duerme sobre el sillón.",
    "Un felino descansa en el sofá.",
    "La inflación anual volvió a bajar.",
]
batch = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True)
with torch.no_grad():
    hidden = model(**batch).last_hidden_state

mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
sentence_embeddings = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
similarities = cosine_similarity(sentence_embeddings.numpy())

sns.heatmap(similarities, annot=True, vmin=0, vmax=1, cmap="crest",
            xticklabels=["gato", "felino", "inflación"],
            yticklabels=["gato", "felino", "inflación"])
plt.title("Similitud coseno entre embeddings de oraciones")
plt.show()

## 5. Cómo cambian las activaciones capa a capa
Cada elemento de `hidden_states` tiene forma `(lote, tokens, dimensiones)`. Graficamos la norma media: una reducción extrema de muchos números a uno, útil solamente para ver que las capas transforman la señal.

In [ ]:
mask_2d = inputs["attention_mask"].bool()
mean_norm = [state.norm(dim=-1)[mask_2d].mean().item() for state in outputs.hidden_states]

plt.plot(range(len(mean_norm)), mean_norm, marker="o", color="#176f62")
plt.xticks(range(len(mean_norm)), ["entrada"] + [f"capa {i}" for i in range(1, len(mean_norm))], rotation=30)
plt.ylabel("Norma L2 media")
plt.title("Magnitud de las activaciones a través del encoder")
plt.grid(alpha=.2)
plt.show()

## 6. Una cabeza de atención
Una matriz de atención indica cuánto pesa cada token de origen para actualizar cada token que consulta, en una capa y cabeza determinadas. **Atención no equivale automáticamente a explicación** del comportamiento del modelo.

In [ ]:
layer, head = -1, 0
attention = outputs.attentions[layer][0, head].numpy()

plt.figure(figsize=(9, 7))
sns.heatmap(attention, xticklabels=tokens, yticklabels=tokens, cmap="mako", square=True)
plt.xlabel("token del que se toma información (key)")
plt.ylabel("token que consulta (query)")
plt.title(f"Atención · última capa, cabeza {head}")
plt.tight_layout()
plt.show()

## Para cerrar
Podés explicar ahora cuatro objetos distintos:

1. **IDs de tokens:** enteros que indexan el vocabulario.
2. **Embeddings de entrada:** vectores iniciales asociados a esos IDs.
3. **Activaciones contextuales:** vectores que cambian al atravesar las capas.
4. **Atenciones:** pesos de combinación para una capa y una cabeza.

**Desafío opcional:** cambiá la oración, la capa y la cabeza; buscá un patrón que te sorprenda y describí qué observás sin afirmar todavía que sea una explicación causal.